In [0]:
from pyspark.sql import functions as F

CATALOG = dbutils.widgets.get("catalog")
RAW_SCHEMA = dbutils.widgets.get("stream_schema")
TITLE = dbutils.widgets.get("title")
SILVER_SCHEMA = f"{RAW_SCHEMA}_silver"

SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.{TITLE}"

In [0]:
EVOLUTION_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.schema_evolution_test"

spark.sql(f"DROP TABLE IF EXISTS {EVOLUTION_TABLE}")

In [0]:
spark.sql(f"""
CREATE TABLE {EVOLUTION_TABLE}
AS SELECT * FROM {SILVER_TABLE}
""")

In [0]:
evolution_df = spark.table(EVOLUTION_TABLE)
display(evolution_df.limit(5))
evolution_df.printSchema()

In [0]:
evolved_data = (
    evolution_df.limit(1).withColumn("quality_score", F.lit(8.7))
)
evolved_data.printSchema()

In [0]:
(
    evolved_data.write
        .format("delta")
        .mode("append")
        .saveAsTable(EVOLUTION_TABLE)
)

In [0]:
(
    evolved_data.write
        .format("delta")
        .option("mergeSchema", "true")
        .mode("append")
        .saveAsTable(EVOLUTION_TABLE)
)

In [0]:
spark.table(EVOLUTION_TABLE).printSchema()

We can also use the second mechanism like:
``` sql
    spark.conf.set(
        "spark.databricks.delta.schema.autoMerge.enabled",
        "true"
    )
```
We can create the new column for example: 
```
    auto_merge_df = (
        spark.table(SILVER_TABLE)
        .limit(1)
        .withColumn("review_score", F.lit(9.2))
    )

    (
    auto_merge_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(SILVER_TABLE)
    )
```

In [0]:
WIDENING_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.widening_test"

spark.sql(f"DROP TABLE IF EXISTS {WIDENING_TABLE}")

test_df = spark.createDataFrame(
    [(1, 100), (2, 200)],
    "id INT, value INT"
)

(
    test_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(WIDENING_TABLE)
)

spark.table(WIDENING_TABLE).printSchema()

In [0]:
spark.sql(f"""
ALTER TABLE {WIDENING_TABLE}
SET TBLPROPERTIES (
    'delta.enableTypeWidening' = 'true'
)
""")

In [0]:
spark.sql(f"""
ALTER TABLE {WIDENING_TABLE}
ALTER COLUMN value TYPE BIGINT
""")

In [0]:
spark.table(WIDENING_TABLE).printSchema()